# few-label-manifold — 5-cell demo

One label per class recovers most of MNIST **if** the representation gives a high-purity graph.
Cells: **data → embedding → graph → 10 labels → result**.

In [ ]:
# 1) DATA + EMBEDDING  (unsupervised contrastive features, shipped in the repo)
import sys, os, numpy as np
sys.path.insert(0, os.path.abspath('../src'))
import representations as R, metrics as M
X, y = R.contrastive()          # (10000, 64) L2-normalised, no labels used to build it
print('embedding:', X.shape, '| classes:', len(np.unique(y)))

In [ ]:
# 2) GRAPH  (k-NN, and the one number that decides everything: edge purity)
W = M.knn_graph(X, k=10)
print(f'edge purity = {M.edge_purity(W, y)*100:.1f}%   bridge edges = {M.bridge_edges(W, y)}')

In [ ]:
# 3) TEN LABELS  (one random point per class) + graph diffusion
rng = np.random.default_rng(1); C = len(np.unique(y))
seeds = np.array([rng.choice(np.where(y == c)[0]) for c in range(C)])
S = M._normalized_operator(W)
acc_di = M.diffusion_spread(S, seeds, y[seeds], y, C)
acc_euc = M.euclid_1nn(X, seeds, y[seeds], y)
print(f'from 10 labels  ->  Euclid 1-NN: {acc_euc*100:.1f}%   |   graph diffusion: {acc_di*100:.1f}%')

In [ ]:
# 4) CEILING  (fully-supervised linear probe, ~7000 labels)
upper = M.linear_upper_bound(X, y)
print(f'10 labels (diffusion): {acc_di*100:.1f}%     7000 labels (linear): {upper*100:.1f}%')

In [ ]:
# 5) RESULT  (the whole point, in one line)
print(f'{acc_di*100:.1f}% of {upper*100:.1f}% achievable, from 700x fewer labels — because the metric is good.')
print('Try a worse metric (raw pixels) and re-run: diffusion collapses. See src/sweep.py for the phase transition.')